# Part 4: Community Detection — Karate Club Network (4.0 points)

We study Wayne Zachary's "karate club" network: a social network of friendships between members of a karate club at a US university. During the observation period, a dispute arose and the club split into two factions of 17 members each. These two factions serve as our ground truth for community detection.

Load the two datasets:
- `../datasets/karate_adjacency.csv` — adjacency matrix $A$ of the graph
- `../datasets/karate_factions.csv` — ground truth faction labels $F$

*(Reference: Newman, "Modularity and community structure in networks", available at `../article-newman.pdf`)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.loadtxt('../datasets/karate_adjacency.csv', delimiter=',')
F = np.loadtxt('../datasets/karate_factions.csv', delimiter=',')

print('Adjacency matrix shape:', A.shape)
print('Factions array shape:', F.shape)
print('Unique faction labels:', np.unique(F))

---
## (a) Definition of modularity (0.5 points)

Define in your own words the notion of **modularity** of a network and how it can be used to split a network into communities. Your description should address whether the modularity depends solely on the structure of the network.

**Answer (a):**

Modularity is a scalar measure $Q \in [-1, 1]$ that quantifies the quality of a partition of a network into communities. It compares the **actual number of edges within communities** to the **expected number of edges** that would exist if the same nodes were connected at random (preserving the degree of each node). Formally:
$$
Q = \frac{1}{4m} \mathbf{s}^\top B\, \mathbf{s}
$$
where $B = A - \frac{\mathbf{d}\mathbf{d}^\top}{2m}$ is the **modularity matrix**, $m$ is the number of edges, $\mathbf{d}$ is the degree vector, and $\mathbf{s} \in \{-1, +1\}^n$ encodes the community assignment.

A high modularity indicates that there are more edges within communities than expected by chance. To split a network into communities, one maximises $Q$ over all possible binary assignments $\mathbf{s}$.

The modularity depends **both on the structure of the network** (via $A$) **and on the proposed partition** (via $\mathbf{s}$) — it is not an intrinsic property of the network alone.

---
## (b) Modularity matrix (0.5 points)

Using the equations seen in class, calculate the modularity matrix $B$ of the graph:
$$B_{ij} = A_{ij} - \frac{d_i d_j}{2m}$$
where $d_i = \sum_j A_{ij}$ is the degree of node $i$ and $m = \frac{1}{2}\sum_{ij} A_{ij}$ is the total number of edges.

In [ ]:
D = np.sum(A, axis=1)   # degree vector
m = np.sum(A) / 2       # total number of edges
B = A - np.outer(D, D) / (2 * m)
print('Modularity matrix B (first 5x5 block):')
print(B[:5, :5])
print('\nShape:', B.shape)
print('Sum of all elements (should be ~0):', np.sum(B))

---
## (c) Modularity Q with ground-truth factions (1.0 point)

Calculate the modularity $Q$ of the graph when fixing the community assignment according to the ground truth factions in array $F$.

**Hint**: encode factions as $s_i \in \{-1, +1\}$ and use $Q = \frac{1}{4m} \mathbf{s}^\top B \mathbf{s}$.

In [ ]:
# Encode factions: F=1 -> s=-1, F=2 -> s=+1  (using 2*(F-1)-1)
S = 2 * (F - 1) - 1
print('Faction encoding S (unique values):', np.unique(S))

Q = S.T @ B @ S / (4 * m)
print('Modularity Q with ground-truth factions:', Q)

---
## (d) Spectral community detection via eigendecomposition (1.5 points)

Based on the eigenvalue/eigenvector decomposition of $B$, split the graph into two communities. Interpret the magnitude of the coordinates of the **leading eigenvector** (i.e. associated with the largest eigenvalue) and explain how it relates to the "faction ambiguity" of each vertex. Are there any nodes for which the split looks more ambiguous?

In [ ]:
w, v = np.linalg.eig(B)

# Sort eigenvalues in descending order
idx = np.argsort(w)[::-1]
w = w[idx]
v = v[:, idx]

print('Top 5 eigenvalues:', np.real(w[:5]))

# Leading eigenvector -> community assignment
v1 = np.real(v[:, 0])
S1 = -1 * np.sign(v1)

print('\nLeading eigenvector coordinates (v1):')
for i, (coord, s) in enumerate(zip(v1, S1)):
    print(f'  Node {i+1:2d}: v1={coord:+.4f}  -> community {int(s):+d}')

In [ ]:
try:
    import networkx as nx
    G = nx.from_numpy_array(A)
    colors = ['red' if s == -1 else 'blue' for s in S1]
    plt.figure(figsize=(8, 6))
    nx.draw_spring(G, node_color=colors, with_labels=True, node_size=300, font_size=8)
    plt.title('Predicted communities (spectral method)')
    plt.show()
except ImportError:
    print('networkx not available; skipping graph plot.')

**Answer (d):**

To maximise $Q = \frac{1}{4m}\mathbf{s}^\top B\mathbf{s}$, we relax the discrete constraint on $\mathbf{s}$ and instead find the eigenvector $\mathbf{v}_1$ corresponding to the **largest eigenvalue** of $B$. The community assignment is then $s_i = \text{sign}(v_{1,i})$.

The **magnitude** $|v_{1,i}|$ reflects the confidence of the assignment for node $i$:
- **Large magnitude**: the node is deeply embedded in one community — unambiguous assignment.
- **Near-zero magnitude**: the node sits close to the boundary between communities — ambiguous assignment.

Nodes whose $|v_{1,i}|$ is close to zero are the most ambiguous and are found at the frontier between the two factions in the karate club graph.

---
## (e) Interpretation when all eigenvalues of B are negative (0.5 points)

What would happen if all eigenvalues of the modularity matrix $B$ were smaller than zero? What would this indicate in terms of the structure of the network?

**Answer (e):**

If all eigenvalues of $B$ are negative, then for any non-trivial assignment vector $\mathbf{s}$:
$$Q = \frac{1}{4m}\mathbf{s}^\top B\mathbf{s} < 0$$

This means **no split of the graph into two communities can yield a positive modularity**. In other words, every partition has fewer within-community edges than expected by chance — there is no viable community structure in the network. The network is essentially **indistinguishable from a random graph** with the same degree sequence (no clustering / community structure exists).